<img src="logo.png" alt="Vegeta" width="240">

# Vegeta Core — revisions, evaluations, NOT RUN

A workspace keeps every design state as an immutable revision and records which analyses ran on it.
Nothing runs by itself: each step below is explicit.

In [ ]:
import shutil
from pathlib import Path
from vegeta import core, talos, mellonia
from vegeta.mellonia.examples import GENERIC_PLA_0_2MM

WS = Path("_runs/core_study")
shutil.rmtree(WS, ignore_errors=True)            # fresh demo workspace
ws = core.Workspace.create(WS, name="bracket study")
bracket = ws.add_design("bracket", "vegeta.dedalus.examples:Bracket")
bracket.record["parameters"]

## 1. A first revision — recorded, nothing generated yet

In [ ]:
r1 = bracket.new_revision(thickness=6.0, note="baseline")
r1, r1.params

In [ ]:
ws.status()

Analyses are refused until geometry exists — nothing is generated automatically:

In [ ]:
r1.run_print('flat', GENERIC_PLA_0_2MM, mellonia.Orientation()).messages

## 2. Generate geometry, then run FEA explicitly

In [ ]:
r1.generate()

In [ ]:
def static(rev):
    return talos.StructuralModel(
        rev.step, "mm-N-MPa",
        talos.Material("Al 6061-T6", youngs_modulus=68900, poissons_ratio=0.33, yield_strength=276,
                       source="nominal handbook values"),
        regions=[talos.SurfacesOnPlane("clamped", "x", -40.0), talos.SurfacesOnPlane("loaded", "x", 40.0)],
        supports=[talos.FixedSupport("clamped")], loads=[talos.Force("loaded", fz=-200.0)],
        mesh_settings=talos.MeshSettings(3.0))

r1.run_fea("static", static)

## 3. Branch a stiffer variant and label it — its analyses show as NOT RUN

In [ ]:
r2 = r1.branch(thickness=8.0, note="stiffer")
r2.label("preferred", note="candidate, needs FEA")
ws.status()

## 4. Run what you decide to run on r2, and print the one you like

In [ ]:
r2.generate()
r2.run_fea("static", static)
r2.run_print("flat", GENERIC_PLA_0_2MM, mellonia.Orientation())
ws.status()

## 5. Everything is on disk and reproducible

In [ ]:
ev = r2.evaluation("fea", "static")
{k: ev.record[k] for k in ("revision", "status", "tool_versions", "started_at")}, ev.record["config"]["loads"]

In [ ]:
for p in sorted((WS / "revisions" / "r2").rglob("*"))[:25]:
    print(p.relative_to(WS))

Results are immutable — re-using a name is refused; use a new name or a new revision:

In [ ]:
r2.run_fea('static', static).messages